# 13 · Capstone — Core Runner
**CAPSTONE** · ⏱ 120 min · 🧪 4 exercises · ⭐ 1100 XP

*Four missions in an original salvage game. Design the state, pick the algorithm, train an agent, and beat par.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "13-capstone-core-runner"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

A derelict station drifts in a dead orbit. Its power cores are still warm. You command a
salvage drone with a limited battery: **collect every core, then reach the extraction hatch**.
Scalding vents and patrolling sentries end the run instantly.

You have already met the environment — your random agent stumbled through it in lesson 01.
Now you will build agents that *master* it, escalating from a lookup table to a neural network.

## 01 · Mission briefing

Try it yourself first. Click the game, then use the keyboard. Getting a feel for the
optimal route makes it much easier to judge whether your agent is any good.

In [ ]:
#@title 🎮 **Interactive** · Core Runner level 1 — play it yourself (click the game, then WASD / arrows)
rlcolab.widget("corerunner", level=1)

| Element | | Reward | Effect |
|---|---|---|---|
| Each step | | −0.1 | battery −1 |
| Bump a wall | `#` | −0.5 | you stay put |
| Energy core | ◆ | **+10** | collected once |
| Extraction hatch | ▢ | **+20** | ends the episode — **locked until every core is collected** |
| Vent | ≋ | −20 | ends the episode |
| Sentry | ▲ | −20 | ends the episode (they move every step, ping-ponging along a track) |
| Battery empty | | — | episode is *truncated* |

Actions: `0 UP`, `1 RIGHT`, `2 DOWN`, `3 LEFT`, `4 WAIT`.

In [ ]:
# 📎 Example from the briefing — run it, change it, break it.
from corerunner import CoreRunner
env = CoreRunner(level=2)               # 1..4
obs, info = env.reset(seed=0)
obs, reward, terminated, truncated, info = env.step(1)
print(env.render())                      # ASCII view
frames = [env.snapshot()]                # JSON frames for rlhud.replay(frames)

> **🔑 KEY IDEA** · **Par**
>
> Every level has a **par** score: the return of a near-optimal run. Your missions are scored
> against it. Matching par means your agent found (essentially) the shortest safe route.

## 02 · What is "the state"?

The Markov property says the state must contain everything needed to predict the future.
In Core Runner that is more than the drone's position:

- **position** — obviously
- **which cores are already collected** — the same cell has very different value before and after
  you grab a core, and the hatch's behaviour depends on it
- **the sentries' phase** — where the sentries are *now* (and so where they will be next step)

`obs="tabular"` (the default) packs exactly these three into one integer in `[0, env.n_states)`:

```text
state = ((row · W + col) · 2ⁿᶜᵒʳᵉˢ + collected_mask) · period + (t mod period)
```

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("13-capstone-core-runner#0", {"q": "Suppose the tabular state only encoded the drone's position, not which cores were collected. What breaks?", "options": ["Nothing — position is enough", "The same position needs different actions before and after a core is collected, so no single Q-value per cell can be right", "Q-learning becomes slower but still converges to the optimal policy"], "answer": 1, "explain": "The process is no longer Markov in that representation. Values before/after collecting a core differ hugely, so a position-only table averages incompatible situations."})

`env.n_states` is 50 on level 1, 392 on level 2 and 31,360 on level 3 (sentry phases multiply the
count). Level 4 generates a **new random wreck every episode**, so a table is hopeless there —
that is where function approximation comes in.

## 03 · Mission 1 — Boot Sequence

Build a tabular Q-learning agent. This is the template you will reuse for the next two
missions, so write it cleanly.

---
### 🧪 Exercise 1 · Mission 1 · Boot Sequence · ⭐ **200 XP**

Write a reusable tabular Q-learning agent for Core Runner.

1. `epsilon_by_episode(ep, episodes, eps_start, eps_end, decay_frac)` — linear decay from `eps_start` to `eps_end` over the first `decay_frac` fraction of training, then flat.
2. `choose_action(Q, s, epsilon, rng)` — ε-greedy over the 5 actions (use `rng.random()` / `rng.integers(5)`; break ties randomly).
3. `train_q(level, episodes, alpha, gamma, eps_start, eps_end, decay_frac, seed)` → returns `(Q, returns)` where `Q` has shape `(env.n_states, 5)` and `returns` is the list of per-episode returns.
4. `greedy_policy(Q)` → a function `obs -> action` that always picks `argmax Q[obs]`.

Remember the Q-learning target: **r** if the episode *terminated*, otherwise **r + γ · maxₐ Q(s′, a)**. A *truncated* episode (battery ran out) still bootstraps.

**Pass:** the greedy policy scores at least **28.0** on level 1 (par 28).

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 13-capstone-core-runner/01-boot-sequence
import numpy as np
import rlhud
from corerunner import CoreRunner, run_episode


def epsilon_by_episode(ep, episodes, eps_start=1.0, eps_end=0.05, decay_frac=0.6):
    """Linear decay over the first decay_frac of training, then constant eps_end."""
    # TODO
    raise NotImplementedError


def choose_action(Q, s, epsilon, rng):
    """ε-greedy with random tie-breaking."""
    # TODO
    raise NotImplementedError


def train_q(level=1, episodes=400, alpha=0.5, gamma=0.99,
            eps_start=1.0, eps_end=0.05, decay_frac=0.6, seed=0):
    """Tabular Q-learning. Returns (Q, list_of_episode_returns)."""
    rng = np.random.default_rng(seed)
    env = CoreRunner(level=level)
    Q = np.zeros((env.n_states, env.n_actions))
    returns = []
    for ep in range(episodes):
        # TODO: one episode of Q-learning
        #   - pick epsilon for this episode
        #   - reset, then loop: choose, step, update Q[s, a], accumulate the return
        #   - stop on terminated or truncated
        raise NotImplementedError
    return Q, returns


def greedy_policy(Q):
    # TODO: return a function obs -> argmax action
    raise NotImplementedError


if __name__ == "__main__":
    Q, returns = train_q(level=1, episodes=400)
    rlhud.plot(returns, label="Q-learning", title="Boot Sequence · training return", smooth=20)
    env = CoreRunner(level=1)
    G, frames, info = run_episode(env, greedy_policy(Q), record=True)
    rlhud.stat("Greedy return", G, good=G >= 28)
    rlhud.stat("Par", 28.0)
    rlhud.stat("States visited", int((np.abs(Q).sum(1) > 0).sum()))
    rlhud.replay(frames, title="Greedy agent · Level 1")

In [ ]:
#@title ✅ **Check** · Mission 1 · Boot Sequence (+200 XP) — tests the code you last ran above
rlcolab.check("13-capstone-core-runner/01-boot-sequence")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

The update is Q[s, a] += alpha * (target - Q[s, a]). Only the `terminated` flag removes the bootstrap term.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Tie-breaking: `best = np.flatnonzero(Q[s] == Q[s].max()); return int(rng.choice(best))`. With an all-zero table, plain argmax would always go UP.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

epsilon_by_episode: `frac = min(1.0, ep / max(1, decay_frac * episodes)); return eps_start + frac * (eps_end - eps_start)`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("13-capstone-core-runner/01-boot-sequence")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 13-capstone-core-runner/01-boot-sequence
# Reference solution
import numpy as np
import rlhud
from corerunner import CoreRunner, run_episode


def epsilon_by_episode(ep, episodes, eps_start=1.0, eps_end=0.05, decay_frac=0.6):
    """Linear decay over the first decay_frac of training, then constant eps_end."""
    frac = min(1.0, ep / max(1, decay_frac * episodes))
    return eps_start + frac * (eps_end - eps_start)


def choose_action(Q, s, epsilon, rng):
    """ε-greedy with random tie-breaking."""
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    best = np.flatnonzero(Q[s] == Q[s].max())
    return int(rng.choice(best))


def train_q(level=1, episodes=400, alpha=0.5, gamma=0.99,
            eps_start=1.0, eps_end=0.05, decay_frac=0.6, seed=0):
    """Tabular Q-learning. Returns (Q, list_of_episode_returns)."""
    rng = np.random.default_rng(seed)
    env = CoreRunner(level=level)
    Q = np.zeros((env.n_states, env.n_actions))
    returns = []
    for ep in range(episodes):
        eps = epsilon_by_episode(ep, episodes, eps_start, eps_end, decay_frac)
        s, _ = env.reset()
        G = 0.0
        while True:
            a = choose_action(Q, s, eps, rng)
            s2, r, terminated, truncated, _ = env.step(a)
            target = r if terminated else r + gamma * Q[s2].max()
            Q[s, a] += alpha * (target - Q[s, a])
            G += r
            s = s2
            if terminated or truncated:
                break
        returns.append(G)
    return Q, returns


def greedy_policy(Q):
    return lambda obs: int(np.argmax(Q[obs]))


if __name__ == "__main__":
    Q, returns = train_q(level=1, episodes=400)
    rlhud.plot(returns, label="Q-learning", title="Boot Sequence · training return", smooth=20)
    env = CoreRunner(level=1)
    G, frames, info = run_episode(env, greedy_policy(Q), record=True)
    rlhud.stat("Greedy return", G, good=G >= 28)
    rlhud.stat("Par", 28.0)
    rlhud.stat("States visited", int((np.abs(Q).sum(1) > 0).sum()))
    rlhud.replay(frames, title="Greedy agent · Level 1")

> **💡 TIP** · **Debugging RL**
>
> If the greedy return is stuck around −10 to −20 the agent has learned to *die quickly* —
> a sign it never discovered the hatch. More exploration, or a longer decay, usually fixes it.
> Always look at a replay before you look at hyperparameters.

## 04 · Mission 2 — Vent Maze

Three cores, a field of vents, and a hatch at the far corner. The reward for the full run is
over 40 steps away and the obvious shortcuts are lethal. Exploration is now the whole game.

In [ ]:
#@title 🎮 **Interactive** · Core Runner level 2 — play it yourself (click the game, then WASD / arrows)
rlcolab.widget("corerunner", level=2)

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("13-capstone-core-runner#1", {"q": "Early in training ε is high and the agent often walks into vents. Why is that actually useful?", "options": ["It isn't — vents should be removed during training", "Dying teaches the agent the large negative value of vent cells, which Q-learning then propagates backwards", "It lowers the learning rate automatically"], "answer": 1, "explain": "Q-learning only knows a vent is bad after experiencing it. Those −20 targets bootstrap into neighbouring states so the greedy policy learns to steer clear."})

---
### 🧪 Exercise 2 · Mission 2 · Vent Maze (1,500-step budget) · ⭐ **250 XP**

Level 2 has three cores and a field of vents. Plain Q-learning *can* solve it — after ~3,000 environment steps. You get **1,500**.

Implement `solve_vent_maze(env, seed, alpha, gamma, n_planning, curve)`:

- `env` is wrapped in a **hard step budget**; `env.steps_left` tells you how many real steps remain, and stepping past zero raises an error.
- Keep running episodes while `env.steps_left > 0` (also stop mid-episode when it hits 0).
- Return a Q table of shape `(env.n_states, 5)`.

The station's physics are **deterministic**, so a model learned from experience is exact. Every real transition you store can be replayed for free — this is **Dyna-Q** from lesson 07.

Tip: decay ε by the *fraction of budget used*, not by episode count.

**Pass:** greedy return ≥ **46** on level 2 for seeds 0, 1 and 2, within budget.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 13-capstone-core-runner/02-vent-maze
import numpy as np
import rlhud
from corerunner import CoreRunner, run_episode


class StepBudget:
    """Wraps an env and refuses to step more than `budget` times in total."""

    def __init__(self, env, budget):
        self.env, self.budget, self.used = env, budget, 0
        self.n_states, self.n_actions = env.n_states, env.n_actions

    @property
    def steps_left(self):
        return self.budget - self.used

    def reset(self, seed=None):
        return self.env.reset(seed=seed)

    def step(self, a):
        if self.used >= self.budget:
            raise RuntimeError(f"Step budget of {self.budget} exhausted — check env.steps_left before stepping")
        self.used += 1
        return self.env.step(a)


def choose_action(Q, s, epsilon, rng):
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    return int(rng.choice(np.flatnonzero(Q[s] == Q[s].max())))


def solve_vent_maze(env, seed=0, alpha=0.5, gamma=0.99, n_planning=30, curve=None):
    """Learn a Q table for level 2 using at most env.steps_left real environment steps.

    Plain Q-learning is NOT sample-efficient enough for this budget. The environment is
    deterministic, so a learned model is perfect: remember (s, a) -> (r, s', terminated)
    and replay it (Dyna-Q). Append each finished episode's return to `curve` if given."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    # TODO: loop over episodes while env.steps_left > 0
    #   - inside an episode also stop when env.steps_left hits 0
    #   - ε-greedy action, real step, Q-learning update
    #   - store the transition in a model, then do n_planning simulated updates
    raise NotImplementedError
    return Q


def greedy_policy(Q):
    return lambda obs: int(np.argmax(Q[obs]))


if __name__ == "__main__":
    BUDGET = 1500
    for n_plan in (0, 10, 30):
        env = StepBudget(CoreRunner(level=2), BUDGET)
        curve = []
        Q = solve_vent_maze(env, seed=0, n_planning=n_plan, curve=curve)
        G, frames, _ = run_episode(CoreRunner(level=2), greedy_policy(Q), record=True)
        rlhud.plot(curve, label=f"n_planning={n_plan}", title="Vent Maze · return per episode (1,500-step budget)")
        rlhud.stat(f"Greedy return (n={n_plan})", G, good=G >= 46)
    rlhud.replay(frames, title="Greedy agent · Level 2")

In [ ]:
#@title ✅ **Check** · Mission 2 · Vent Maze (1,500-step budget) (+250 XP) — tests the code you last ran above
rlcolab.check("13-capstone-core-runner/02-vent-maze")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Keep `model[(s, a)] = (r, s2, terminated)` and a list of the keys you have seen so you can sample one uniformly with rng.integers(len(keys)).


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

After each real Q-update, do n_planning updates: pick a random seen (s, a), look up (r, s2, term) and apply the same Q-learning update to it.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

eps = max(0.05, 1 - used / (0.5 * budget)) where used = budget - env.steps_left. With n_planning=30 you should reach 47.6.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("13-capstone-core-runner/02-vent-maze")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 13-capstone-core-runner/02-vent-maze
# Reference solution
import numpy as np
import rlhud
from corerunner import CoreRunner, run_episode


class StepBudget:
    """Wraps an env and refuses to step more than `budget` times in total."""

    def __init__(self, env, budget):
        self.env, self.budget, self.used = env, budget, 0
        self.n_states, self.n_actions = env.n_states, env.n_actions

    @property
    def steps_left(self):
        return self.budget - self.used

    def reset(self, seed=None):
        return self.env.reset(seed=seed)

    def step(self, a):
        if self.used >= self.budget:
            raise RuntimeError(f"Step budget of {self.budget} exhausted — check env.steps_left before stepping")
        self.used += 1
        return self.env.step(a)


def choose_action(Q, s, epsilon, rng):
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    return int(rng.choice(np.flatnonzero(Q[s] == Q[s].max())))


def solve_vent_maze(env, seed=0, alpha=0.5, gamma=0.99, n_planning=30, curve=None):
    """Dyna-Q: learn from each real step, then replay n_planning imagined steps from a learned model.

    env has a hard step budget (env.steps_left). Returns the Q table."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    model = {}          # (s, a) -> (r, s', terminated) — the level is deterministic
    seen = []           # list of (s, a) keys for O(1) random sampling
    budget = env.steps_left
    while env.steps_left > 0:
        s, _ = env.reset()
        G = 0.0
        while env.steps_left > 0:
            # ε decays with the fraction of the budget used, not with episodes
            eps = max(0.05, 1.0 - (budget - env.steps_left) / (0.5 * budget))
            a = choose_action(Q, s, eps, rng)
            s2, r, term, trunc, _ = env.step(a)
            G += r
            Q[s, a] += alpha * ((r if term else r + gamma * Q[s2].max()) - Q[s, a])
            if (s, a) not in model:
                seen.append((s, a))
            model[(s, a)] = (r, s2, term)
            for _ in range(n_planning):      # planning: free updates from the model
                ps, pa = seen[rng.integers(len(seen))]
                pr, ps2, pterm = model[(ps, pa)]
                Q[ps, pa] += alpha * ((pr if pterm else pr + gamma * Q[ps2].max()) - Q[ps, pa])
            s = s2
            if term or trunc:
                if curve is not None:
                    curve.append(G)
                break
    return Q


def greedy_policy(Q):
    return lambda obs: int(np.argmax(Q[obs]))


if __name__ == "__main__":
    BUDGET = 1500
    for n_plan in (0, 10, 30):
        env = StepBudget(CoreRunner(level=2), BUDGET)
        curve = []
        Q = solve_vent_maze(env, seed=0, n_planning=n_plan, curve=curve)
        G, frames, _ = run_episode(CoreRunner(level=2), greedy_policy(Q), record=True)
        rlhud.plot(curve, label=f"n_planning={n_plan}", title="Vent Maze · return per episode (1,500-step budget)")
        rlhud.stat(f"Greedy return (n={n_plan})", G, good=G >= 46)
    rlhud.replay(frames, title="Greedy agent · Level 2")

## 05 · Mission 3 — Patrol Deck

Two sentries sweep the corridors. Sometimes the right move is to **wait**. Because the tabular
state includes the sentry phase, plain Q-learning can still solve this level exactly — but the
table is now 80× bigger, so learning efficiency matters.

In [ ]:
#@title 🎮 **Interactive** · Core Runner level 3 — play it yourself (click the game, then WASD / arrows)
rlcolab.widget("corerunner", level=3)

> **ℹ️ NOTE** · **Algorithm choice**
>
> Anything tabular from the course is fair game: Q-learning, SARSA, Expected SARSA, n-step,
> Dyna-Q. Dyna-Q is a particularly good fit — the environment is deterministic, so a learned
> model is perfect and every planning step is free experience.

---
### 🧪 Exercise 3 · Mission 3 · Patrol Deck (design the state) · ⭐ **250 XP**

On this mission you don't get a ready-made tabular state — you **design it**.

1. `encode_state(env)` → one integer that captures everything needed to predict the future, built from `env.pos`, `env.collected` and the **sentry phase** `env.t % env.period`.
2. `n_states_for(env)` → how many values your encoder can return.
3. Finish the Q-learning loop in `train_patrol` (it calls your encoder after every step).

Run the demo first: it trains the same agent twice — once with a *position-only* state that ignores the sentries, once with yours. Watch both replays.

**Pass:** your encoder is collision-free and in range for every (position, cores, phase) combination, and the greedy agent scores ≥ **44** (par) on seeds 0 and 1.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 13-capstone-core-runner/03-patrol-deck
import numpy as np
import rlhud
from corerunner import CoreRunner


# ---------------------------------------------------------------- state design
# The env exposes its raw internals:
#   env.pos            (row, col) of the drone
#   env.collected      list[bool], one per core
#   env.t              steps taken this episode
#   env.period         sentries repeat their motion every `period` steps
#   env.h, env.w       grid size;  len(env.core_list) = number of cores

def n_states_for(env):
    """How many distinct encoded states encode_state can return."""
    # TODO
    raise NotImplementedError


def encode_state(env):
    """Pack everything that matters for the future into ONE integer in [0, n_states_for(env))."""
    # TODO: combine position, which cores are collected (a bitmask) and the sentry phase
    raise NotImplementedError


def encode_position_only(env):
    """A deliberately BROKEN state (ignores sentries) — for comparison in the demo."""
    r, c = env.pos
    mask = sum(int(g) << i for i, g in enumerate(env.collected))
    return (r * env.w + c) * (2 ** len(env.core_list)) + mask


# ---------------------------------------------------------------- learning
def train_patrol(episodes=3000, seed=0, encoder=encode_state, n_states=None,
                 alpha=0.5, gamma=0.99, curve=None):
    """Q-learning on level 3 using YOUR state encoder (env observations are ignored)."""
    rng = np.random.default_rng(seed)
    env = CoreRunner(level=3)
    Q = np.zeros((n_states or n_states_for(env), env.n_actions))
    for ep in range(episodes):
        eps = max(0.05, 1.0 - ep / (0.6 * episodes))
        env.reset()
        s, G = encoder(env), 0.0
        # TODO: the Q-learning episode loop — after env.step(a), compute s2 = encoder(env)
        raise NotImplementedError
        if curve is not None:
            curve.append(G)
    return Q


def play_greedy(Q, encoder=encode_state, record=False):
    env = CoreRunner(level=3)
    env.reset()
    frames, G = [env.snapshot()], 0.0
    while True:
        _, r, term, trunc, info = env.step(int(np.argmax(Q[encoder(env)])))
        G += r
        if record:
            frames.append(env.snapshot())
        if term or trunc:
            return G, info["event"], frames


if __name__ == "__main__":
    env = CoreRunner(level=3)
    for name, enc, n in [("position + cores (no sentry phase)", encode_position_only, env.h * env.w * 8),
                         ("full Markov state", encode_state, n_states_for(env))]:
        curve = []
        Q = train_patrol(3000, seed=0, encoder=enc, n_states=n, curve=curve)
        G, event, frames = play_greedy(Q, enc, record=True)
        rlhud.plot(curve, label=name, title="Patrol Deck · training return", smooth=50)
        rlhud.stat(name, f"{G:.1f} ({event})")
        rlhud.replay(frames, title=f"Greedy · {name}")

In [ ]:
#@title ✅ **Check** · Mission 3 · Patrol Deck (design the state) (+250 XP) — tests the code you last ran above
rlcolab.check("13-capstone-core-runner/03-patrol-deck")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Think of mixed-radix numbers: index = ((cell · n_masks) + mask) · period + phase, with cell = row · W + col and n_masks = 2 ** n_cores.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

The cores bitmask: mask = sum(int(got) << i for i, got in enumerate(env.collected)). Phase is env.t % env.period — not env.t itself, or the table would be infinite.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

n_states_for(env) = env.h * env.w * 2 ** len(env.core_list) * env.period. The loop body is the same as Mission 1, except s2 = encoder(env) instead of the observation.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("13-capstone-core-runner/03-patrol-deck")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 13-capstone-core-runner/03-patrol-deck
# Reference solution
import numpy as np
import rlhud
from corerunner import CoreRunner


# ---------------------------------------------------------------- state design
# The env exposes its raw internals:
#   env.pos            (row, col) of the drone
#   env.collected      list[bool], one per core
#   env.t              steps taken this episode
#   env.period         sentries repeat their motion every `period` steps
#   env.h, env.w       grid size;  len(env.core_list) = number of cores

def n_states_for(env):
    """How many distinct encoded states encode_state can return."""
    return env.h * env.w * (2 ** len(env.core_list)) * env.period


def encode_state(env):
    """A Markov state: position x collected-cores bitmask x sentry phase, packed into one int."""
    r, c = env.pos
    mask = 0
    for i, got in enumerate(env.collected):
        mask |= int(got) << i
    cell = r * env.w + c
    return (cell * (2 ** len(env.core_list)) + mask) * env.period + (env.t % env.period)


def encode_position_only(env):
    """A deliberately BROKEN state (ignores sentries) — for comparison in the demo."""
    r, c = env.pos
    mask = sum(int(g) << i for i, g in enumerate(env.collected))
    return (r * env.w + c) * (2 ** len(env.core_list)) + mask


# ---------------------------------------------------------------- learning
def train_patrol(episodes=3000, seed=0, encoder=encode_state, n_states=None,
                 alpha=0.5, gamma=0.99, curve=None):
    """Q-learning on level 3 using YOUR state encoder (env observations are ignored)."""
    rng = np.random.default_rng(seed)
    env = CoreRunner(level=3)
    Q = np.zeros((n_states or n_states_for(env), env.n_actions))
    for ep in range(episodes):
        eps = max(0.05, 1.0 - ep / (0.6 * episodes))
        env.reset()
        s, G = encoder(env), 0.0
        while True:
            if rng.random() < eps:
                a = int(rng.integers(5))
            else:
                a = int(rng.choice(np.flatnonzero(Q[s] == Q[s].max())))
            _, r, term, trunc, _ = env.step(a)
            s2 = encoder(env)
            Q[s, a] += alpha * ((r if term else r + gamma * Q[s2].max()) - Q[s, a])
            s, G = s2, G + r
            if term or trunc:
                break
        if curve is not None:
            curve.append(G)
    return Q


def play_greedy(Q, encoder=encode_state, record=False):
    env = CoreRunner(level=3)
    env.reset()
    frames, G = [env.snapshot()], 0.0
    while True:
        _, r, term, trunc, info = env.step(int(np.argmax(Q[encoder(env)])))
        G += r
        if record:
            frames.append(env.snapshot())
        if term or trunc:
            return G, info["event"], frames


if __name__ == "__main__":
    env = CoreRunner(level=3)
    for name, enc, n in [("position + cores (no sentry phase)", encode_position_only, env.h * env.w * 8),
                         ("full Markov state", encode_state, n_states_for(env))]:
        curve = []
        Q = train_patrol(3000, seed=0, encoder=enc, n_states=n, curve=curve)
        G, event, frames = play_greedy(Q, enc, record=True)
        rlhud.plot(curve, label=name, title="Patrol Deck · training return", smooth=50)
        rlhud.stat(name, f"{G:.1f} ({event})")
        rlhud.replay(frames, title=f"Greedy · {name}")

## 06 · Mission 4 — Deep Salvage

Every episode is a freshly generated 6×6 wreck. You cannot memorise a map; you have to learn a
*skill*. Switch to `obs="vector"`: a 133-number observation made of

- an egocentric **5×5 window** around the drone with five channels (wall, vent, core, sentry, exit) — 125 numbers
- **8 globals**: the relative position of the current target (nearest core, or the hatch once
  unlocked), four direction flags, whether the hatch is open, and the battery fraction

That is exactly the setting for a **Deep Q-Network**: a neural network maps the observation to
five Q-values, trained from a replay buffer against a slowly-updated target network.

In [ ]:
#@title 🎮 **Interactive** · Core Runner level 4 — play it yourself (click the game, then WASD / arrows)
rlcolab.widget("corerunner", level=4)

This mission works like real RL research: you **train in Run** (as long as you like — raise the
time limit in Settings → System if you need to), save the weights to your workspace, and
**Check evaluates your saved agent** on 50 wrecks it has never seen.

---
### 🧪 Exercise 4 · Mission 4 · Deep Salvage (DQN) · ⭐ **400 XP**

Every episode is a brand-new procedurally generated wreck. Train a **Deep Q-Network** that generalises.

The network (`MLP` with Adam), the n-step `ReplayBuffer`, the training loop and the evaluation are provided. You write the RL core:

1. `epsilon_at(step, total_steps, explore_frac, eps_end)` — linear exploration schedule.
2. `nstep_return(rewards, gamma)` → `(Σ γᵏ rₖ, γⁿ)`: the n-step return and the discount for its bootstrap.
3. `double_dqn_targets(online, target, R, S2, D, disc)` → `R + disc · (1 − D) · Q_target(s′, argmaxₐ Q_online(s′, a))`.
4. `huber_grad(pred, y)` → `clip(pred − y, −1, 1) / N`.
5. The gradient step inside `train()`.

**Workflow:** press **Run** to train (≈1–2 min on a laptop; raise the time limit in Settings → System if needed). It saves `deep_salvage.npz` to your workspace. Then press **Check**, which unit-tests your functions and evaluates the *saved* agent on 50 wrecks it has never seen.

**Pass:** mean return ≥ **15** and extraction on ≥ **45%** of unseen wrecks. A hand-coded autopilot gets ~80% — can you beat it?

> 🏋️ **Train, then check.** Running the next cell trains your agent and saves its weights into `RewardSignal/workspace/` on your Drive. **✅ Check** then tests your functions and evaluates the *saved* agent — no retraining. Training only needs the CPU runtime.

In [ ]:
# ✏️ EXERCISE 13-capstone-core-runner/04-deep-salvage
import os
import time
import numpy as np
import rlhud
from corerunner import CoreRunner

WEIGHTS = "deep_salvage.npz"          # saved in your workspace folder


# ================================================================ provided: network + optimiser
class MLP:
    """Fully-connected ReLU network with a linear output layer, trained with Adam."""

    def __init__(self, sizes, rng):
        self.W = [rng.normal(0, np.sqrt(2 / a), (a, b)) for a, b in zip(sizes, sizes[1:])]
        self.b = [np.zeros(b) for b in sizes[1:]]
        self._m = [np.zeros_like(p) for p in self.params()]
        self._v = [np.zeros_like(p) for p in self.params()]
        self._t = 0

    def params(self):
        return self.W + self.b

    def forward(self, x):
        self._acts = [x]
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            x = x @ W + b
            if i < len(self.W) - 1:
                x = np.maximum(x, 0)
            self._acts.append(x)
        return x

    def backward(self, grad_out, lr):
        """Backprop dLoss/dOutput through the last forward() call and take one Adam step."""
        gW, gb, g = [], [], grad_out
        for i in reversed(range(len(self.W))):
            gW.insert(0, self._acts[i].T @ g)
            gb.insert(0, g.sum(0))
            if i > 0:
                g = (g @ self.W[i].T) * (self._acts[i] > 0)
        self._t += 1
        for k, (p, gr) in enumerate(zip(self.params(), gW + gb)):
            self._m[k] = 0.9 * self._m[k] + 0.1 * gr
            self._v[k] = 0.999 * self._v[k] + 0.001 * gr * gr
            p -= lr * (self._m[k] / (1 - 0.9 ** self._t)) / (np.sqrt(self._v[k] / (1 - 0.999 ** self._t)) + 1e-8)

    def copy_from(self, other):
        self.W = [w.copy() for w in other.W]
        self.b = [b.copy() for b in other.b]

    def save(self, path):
        np.savez(path, *self.W, *self.b)

    @classmethod
    def load(cls, path):
        data = np.load(path)
        arrs = [data[f"arr_{i}"] for i in range(len(data.files))]
        net = cls.__new__(cls)
        half = len(arrs) // 2
        net.W, net.b = arrs[:half], arrs[half:]
        return net


class ReplayBuffer:
    """Stores n-step transitions (s, a, R_n, s_n, done, discount=γ^k)."""

    def __init__(self, capacity, obs_size):
        self.S = np.zeros((capacity, obs_size), np.float32)
        self.S2 = np.zeros((capacity, obs_size), np.float32)
        self.A = np.zeros(capacity, np.int64)
        self.R = np.zeros(capacity)
        self.D = np.zeros(capacity)
        self.G = np.zeros(capacity)
        self.cap, self.i, self.n = capacity, 0, 0

    def add(self, s, a, r, s2, done, disc):
        j = self.i
        self.S[j], self.A[j], self.R[j], self.S2[j], self.D[j], self.G[j] = s, a, r, s2, done, disc
        self.i = (self.i + 1) % self.cap
        self.n = min(self.n + 1, self.cap)

    def sample(self, k, rng):
        idx = rng.integers(self.n, size=k)
        return self.S[idx], self.A[idx], self.R[idx], self.S2[idx], self.D[idx], self.G[idx]


# ================================================================ your part
def epsilon_at(step, total_steps, explore_frac=0.4, eps_end=0.05):
    """Linear decay from 1.0 to eps_end over the first explore_frac of training."""
    # TODO
    raise NotImplementedError


def nstep_return(rewards, gamma):
    """Discounted sum of a short list of rewards, and the discount to apply to the bootstrap.
    nstep_return([r0, r1, r2], γ) -> (r0 + γ r1 + γ² r2, γ³)"""
    # TODO
    raise NotImplementedError


def double_dqn_targets(online, target, R, S2, D, disc):
    """y = R + disc · (1 − D) · Q_target(s', argmaxₐ Q_online(s', a))."""
    # TODO: pick a* with the ONLINE net, evaluate it with the TARGET net
    raise NotImplementedError


def huber_grad(pred, y, delta=1.0):
    """d/dpred of the Huber loss, averaged over the batch: clip(pred − y, −δ, δ) / N."""
    # TODO
    raise NotImplementedError


def train(total_steps=60_000, seed=0, hidden=128, gamma=0.97, lr=5e-4, n_step=3,
          batch=64, train_every=2, target_every=1000, chart=None):
    """Train a DQN on procedurally generated wrecks. Returns the online network."""
    rng = np.random.default_rng(seed)
    env = CoreRunner(level=4, obs="vector", seed=seed)
    q = MLP([env.obs_size, hidden, 5], rng)
    q_target = MLP([env.obs_size, hidden, 5], rng)
    q_target.copy_from(q)
    buf = ReplayBuffer(60_000, env.obs_size)
    obs, _ = env.reset()
    pending, ep_return = [], 0.0          # last n (obs, action, reward) for n-step returns
    for step in range(total_steps):
        if rng.random() < epsilon_at(step, total_steps):
            a = int(rng.integers(5))
        else:
            a = int(np.argmax(q.forward(obs[None])[0]))
        obs2, r, term, trunc, _ = env.step(a)
        ep_return += r
        pending.append((obs, a, r))
        if len(pending) == n_step:        # oldest transition now has its full n-step return
            R, disc = nstep_return([p[2] for p in pending], gamma)
            buf.add(pending[0][0], pending[0][1], R, obs2, float(term), disc)
            pending.pop(0)
        obs = obs2
        if term or trunc:                 # flush shorter returns at the end of an episode
            while pending:
                R, disc = nstep_return([p[2] for p in pending], gamma)
                buf.add(pending[0][0], pending[0][1], R, obs2, float(term), disc)
                pending.pop(0)
            if chart is not None:
                chart.add("episode return", ep_return)
            obs, _ = env.reset()
            ep_return = 0.0
        if buf.n >= 2000 and step % train_every == 0:
            S, A, R, S2, D, G = buf.sample(batch, rng)
            # TODO: one gradient step
            #   1. targets y with double_dqn_targets (no gradient flows through y)
            #   2. qs = q.forward(S) — only the taken actions' Q-values get a gradient
            #   3. grad = zeros like qs; grad[rows, A] = huber_grad(...)
            #   4. q.backward(grad, lr)
            raise NotImplementedError
        if step % target_every == 0:
            q_target.copy_from(q)
        rlhud.progress(step, total_steps, "Training DQN")
    return q


def load_agent(path=WEIGHTS):
    """Load saved weights and return a greedy policy obs -> action."""
    net = MLP.load(path)
    return lambda obs: int(np.argmax(net.forward(np.asarray(obs, np.float32)[None])[0]))


# ================================================================ evaluation (provided)
def evaluate(policy, episodes=50, seed=50_000, record=0):
    env = CoreRunner(level=4, obs="vector")
    returns, escapes, replays = [], 0, []
    for i in range(episodes):
        obs, _ = env.reset(seed=seed + i)
        frames, G = [env.snapshot()], 0.0
        while True:
            obs, r, term, trunc, info = env.step(policy(obs))
            G += r
            if i < record:
                frames.append(env.snapshot())
            if term or trunc:
                break
        returns.append(G)
        escapes += info["event"] == "escaped"
        if i < record:
            replays.append(frames)
    return float(np.mean(returns)), escapes / episodes, replays


if __name__ == "__main__":
    t0 = time.time()
    chart = rlhud.chart("Deep Salvage · training return", smooth=100)
    net = train(total_steps=80_000, seed=0, chart=chart)
    chart.flush()
    net.save(WEIGHTS)
    print(f"trained in {time.time() - t0:.0f}s, saved weights to {os.path.abspath(WEIGHTS)}")
    mean, success, replays = evaluate(load_agent(), record=3)
    rlhud.stat("Mean return (50 unseen wrecks)", mean)
    rlhud.stat("Extraction rate", f"{success:.0%}", good=success >= 0.45)
    for k, frames in enumerate(replays):
        rlhud.replay(frames, title=f"Unseen wreck #{k + 1}")

In [ ]:
#@title ✅ **Check** · Mission 4 · Deep Salvage (DQN) (+400 XP) — tests the code you last ran above
rlcolab.check("13-capstone-core-runner/04-deep-salvage")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Double DQN decouples selection from evaluation: a_star = online.forward(S2).argmax(1); q_next = target.forward(S2)[np.arange(len(S2)), a_star].


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Only the Q-values of the actions actually taken receive gradient: rows = np.arange(batch); grad = np.zeros_like(qs); grad[rows, A] = huber_grad(qs[rows, A], y).


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

If training collapses: lower lr (3e-4), longer exploration (explore_frac 0.5), or more total_steps (100k). Watch the replays of failures — getting stuck against walls means it needs more exploration.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("13-capstone-core-runner/04-deep-salvage")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 13-capstone-core-runner/04-deep-salvage
# Reference solution
import os
import time
import numpy as np
import rlhud
from corerunner import CoreRunner

WEIGHTS = "deep_salvage.npz"          # saved in your workspace folder


# ================================================================ provided: network + optimiser
class MLP:
    """Fully-connected ReLU network with a linear output layer, trained with Adam."""

    def __init__(self, sizes, rng):
        self.W = [rng.normal(0, np.sqrt(2 / a), (a, b)) for a, b in zip(sizes, sizes[1:])]
        self.b = [np.zeros(b) for b in sizes[1:]]
        self._m = [np.zeros_like(p) for p in self.params()]
        self._v = [np.zeros_like(p) for p in self.params()]
        self._t = 0

    def params(self):
        return self.W + self.b

    def forward(self, x):
        self._acts = [x]
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            x = x @ W + b
            if i < len(self.W) - 1:
                x = np.maximum(x, 0)
            self._acts.append(x)
        return x

    def backward(self, grad_out, lr):
        """Backprop dLoss/dOutput through the last forward() call and take one Adam step."""
        gW, gb, g = [], [], grad_out
        for i in reversed(range(len(self.W))):
            gW.insert(0, self._acts[i].T @ g)
            gb.insert(0, g.sum(0))
            if i > 0:
                g = (g @ self.W[i].T) * (self._acts[i] > 0)
        self._t += 1
        for k, (p, gr) in enumerate(zip(self.params(), gW + gb)):
            self._m[k] = 0.9 * self._m[k] + 0.1 * gr
            self._v[k] = 0.999 * self._v[k] + 0.001 * gr * gr
            p -= lr * (self._m[k] / (1 - 0.9 ** self._t)) / (np.sqrt(self._v[k] / (1 - 0.999 ** self._t)) + 1e-8)

    def copy_from(self, other):
        self.W = [w.copy() for w in other.W]
        self.b = [b.copy() for b in other.b]

    def save(self, path):
        np.savez(path, *self.W, *self.b)

    @classmethod
    def load(cls, path):
        data = np.load(path)
        arrs = [data[f"arr_{i}"] for i in range(len(data.files))]
        net = cls.__new__(cls)
        half = len(arrs) // 2
        net.W, net.b = arrs[:half], arrs[half:]
        return net


class ReplayBuffer:
    """Stores n-step transitions (s, a, R_n, s_n, done, discount=γ^k)."""

    def __init__(self, capacity, obs_size):
        self.S = np.zeros((capacity, obs_size), np.float32)
        self.S2 = np.zeros((capacity, obs_size), np.float32)
        self.A = np.zeros(capacity, np.int64)
        self.R = np.zeros(capacity)
        self.D = np.zeros(capacity)
        self.G = np.zeros(capacity)
        self.cap, self.i, self.n = capacity, 0, 0

    def add(self, s, a, r, s2, done, disc):
        j = self.i
        self.S[j], self.A[j], self.R[j], self.S2[j], self.D[j], self.G[j] = s, a, r, s2, done, disc
        self.i = (self.i + 1) % self.cap
        self.n = min(self.n + 1, self.cap)

    def sample(self, k, rng):
        idx = rng.integers(self.n, size=k)
        return self.S[idx], self.A[idx], self.R[idx], self.S2[idx], self.D[idx], self.G[idx]


# ================================================================ your part
def epsilon_at(step, total_steps, explore_frac=0.4, eps_end=0.05):
    """Linear decay from 1.0 to eps_end over the first explore_frac of training."""
    return max(eps_end, 1.0 - (1.0 - eps_end) * step / (explore_frac * total_steps))


def nstep_return(rewards, gamma):
    """Discounted sum of a short list of rewards, and the discount to apply to the bootstrap.
    nstep_return([r0, r1, r2], γ) -> (r0 + γ r1 + γ² r2, γ³)"""
    G = 0.0
    for k, r in enumerate(rewards):
        G += gamma ** k * r
    return G, gamma ** len(rewards)


def double_dqn_targets(online, target, R, S2, D, disc):
    """y = R + disc · (1 − D) · Q_target(s', argmaxₐ Q_online(s', a))."""
    a_star = online.forward(S2).argmax(axis=1)
    q_next = target.forward(S2)[np.arange(len(S2)), a_star]
    return R + disc * (1.0 - D) * q_next


def huber_grad(pred, y, delta=1.0):
    """d/dpred of the Huber loss, averaged over the batch: clip(pred − y, −δ, δ) / N."""
    return np.clip(pred - y, -delta, delta) / len(pred)


def train(total_steps=60_000, seed=0, hidden=128, gamma=0.97, lr=5e-4, n_step=3,
          batch=64, train_every=2, target_every=1000, chart=None):
    """Train a DQN on procedurally generated wrecks. Returns the online network."""
    rng = np.random.default_rng(seed)
    env = CoreRunner(level=4, obs="vector", seed=seed)
    q = MLP([env.obs_size, hidden, 5], rng)
    q_target = MLP([env.obs_size, hidden, 5], rng)
    q_target.copy_from(q)
    buf = ReplayBuffer(60_000, env.obs_size)
    obs, _ = env.reset()
    pending, ep_return = [], 0.0          # last n (obs, action, reward) for n-step returns
    for step in range(total_steps):
        if rng.random() < epsilon_at(step, total_steps):
            a = int(rng.integers(5))
        else:
            a = int(np.argmax(q.forward(obs[None])[0]))
        obs2, r, term, trunc, _ = env.step(a)
        ep_return += r
        pending.append((obs, a, r))
        if len(pending) == n_step:        # oldest transition now has its full n-step return
            R, disc = nstep_return([p[2] for p in pending], gamma)
            buf.add(pending[0][0], pending[0][1], R, obs2, float(term), disc)
            pending.pop(0)
        obs = obs2
        if term or trunc:                 # flush shorter returns at the end of an episode
            while pending:
                R, disc = nstep_return([p[2] for p in pending], gamma)
                buf.add(pending[0][0], pending[0][1], R, obs2, float(term), disc)
                pending.pop(0)
            if chart is not None:
                chart.add("episode return", ep_return)
            obs, _ = env.reset()
            ep_return = 0.0
        if buf.n >= 2000 and step % train_every == 0:
            S, A, R, S2, D, G = buf.sample(batch, rng)
            y = double_dqn_targets(q, q_target, R, S2, D, G)
            qs = q.forward(S)
            grad = np.zeros_like(qs)
            grad[np.arange(batch), A] = huber_grad(qs[np.arange(batch), A], y)
            q.backward(grad, lr)
        if step % target_every == 0:
            q_target.copy_from(q)
        rlhud.progress(step, total_steps, "Training DQN")
    return q


def load_agent(path=WEIGHTS):
    """Load saved weights and return a greedy policy obs -> action."""
    net = MLP.load(path)
    return lambda obs: int(np.argmax(net.forward(np.asarray(obs, np.float32)[None])[0]))


# ================================================================ evaluation (provided)
def evaluate(policy, episodes=50, seed=50_000, record=0):
    env = CoreRunner(level=4, obs="vector")
    returns, escapes, replays = [], 0, []
    for i in range(episodes):
        obs, _ = env.reset(seed=seed + i)
        frames, G = [env.snapshot()], 0.0
        while True:
            obs, r, term, trunc, info = env.step(policy(obs))
            G += r
            if i < record:
                frames.append(env.snapshot())
            if term or trunc:
                break
        returns.append(G)
        escapes += info["event"] == "escaped"
        if i < record:
            replays.append(frames)
    return float(np.mean(returns)), escapes / episodes, replays


if __name__ == "__main__":
    t0 = time.time()
    chart = rlhud.chart("Deep Salvage · training return", smooth=100)
    net = train(total_steps=80_000, seed=0, chart=chart)
    chart.flush()
    net.save(WEIGHTS)
    print(f"trained in {time.time() - t0:.0f}s, saved weights to {os.path.abspath(WEIGHTS)}")
    mean, success, replays = evaluate(load_agent(), record=3)
    rlhud.stat("Mean return (50 unseen wrecks)", mean)
    rlhud.stat("Extraction rate", f"{success:.0%}", good=success >= 0.45)
    for k, frames in enumerate(replays):
        rlhud.replay(frames, title=f"Unseen wreck #{k + 1}")

> **⚠️ WARNING** · **Deep RL is noisy**
>
> Two runs with different seeds can differ a lot. If training collapses, try: a smaller learning
> rate, a slower ε decay, Double-DQN targets, gradient/TD-error clipping, or simply a different seed.
> Plot the training return and watch a replay of a failure case before changing anything.

## 07 · Debrief

You took one environment from a random walk to a trained neural agent:

| Mission | Representation | Method | Why it works |
|---|---|---|---|
| 1–2 | exact tabular state | Q-learning + ε schedule | small, Markov, deterministic |
| 3 | tabular with sentry phase | Q-learning / Dyna-Q | time folded into the state keeps it Markov |
| 4 | local features | DQN | generalises across layouts it has never seen |

Ideas to push further on your own:

- **Reward shaping** with a potential function Φ(s) (e.g. negative distance to target) —
  shaping of the form γΦ(s') − Φ(s) provably keeps the optimal policy unchanged.
- A **policy-gradient** agent (REINFORCE / PPO) on level 4 — compare sample efficiency with DQN.
- Use `obs="grid"` with a small convolutional network if you install PyTorch.
- Make your own levels: `CoreRunner(layout=[...])` accepts any ASCII map.

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("13-capstone-core-runner")